In [ ]:
#!pip install soundfile

In [ ]:
import importlib
from dataclasses import dataclass
from typing import Any, Dict, Optional
import numpy as np
import math
from tqdm import tqdm
import io
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from itertools import islice

import library.utils.general
importlib.reload(library.utils.general)
from library.utils.general import *
import library.utils.tokenization
importlib.reload(library.utils.tokenization)
from library.utils.tokenization import *

import library.model.audio
importlib.reload(library.model.audio)
from library.model.audio import *

In [ ]:
try:
    from datasets import load_dataset, Audio as HFAudio
    _HAS_HF = True
except Exception:
    _HAS_HF = False
try:
    import soundfile as sf
    _HAS_SF = True
except Exception:
    sf = None
    _HAS_SF = False
try:
    import audioread
    _HAS_AR = True
except Exception:
    audioread = None
    _HAS_AR = False
try:
    import torchaudio
    _HAS_TA = True
except Exception:
    torchaudio = None
    _HAS_TA = False

In [ ]:
DEFAULT_TEXT_LENGTH_TOKENS = 64
DEFAULT_AUDIO_LENGTH_SECONDS = 4
SAMPLE_RATE = 16000

vocab = TextVocabulary()

vocab.add_tokens([str(t).upper() for t in [chr(i) for i in range(32, 127)]])
vocab.add_tokens([str(t).upper() for t in [
    "un", "re", "in", "im", "il", "ir", "dis", "en", "em", "non",
    "in", "im", "over", "mis", "sub", "pre", "inter", "fore", "de", "trans",
    "super", "semi", "anti", "mid", "under", "hyper", "auto", "bio", "micro",
    "macro", "mini", "mono", "multi", "poly", "tri", "uni", "a", "ab", "ad",
    "ambi", "amphi", "ante", "anti", "arch", "be", "bi", "circum", "co", "col",
    "com", "con", "contra", "counter", "de", "dia", "epi", "ex", "extra",
    "fore", "hetero", "homo", "hyper", "hypo", "intra", "inter", "intro",
    "mal", "mis", "ob", "omni", "para", "peri", "post", "pre", "pro", "pseudo",
    "retro", "semi", "sub", "super", "syn", "sym", "tele", "trans", "ultra", "vice"
]])
vocab.add_tokens([str(t).upper() for t in [
    "able", "ible", "al", "ial", "ed", "en", "er", "est", "ful", "ic", "ing",
    "ion", "tion", "ation", "ition", "ment", "ness", "ous", "eous", "ious",
    "less", "ly", "ise", "ize", "fy", "ish", "ism", "ist", "ity", "ty", "ment",
    "ness", "ship", "sion", "tion", "acy", "al", "ance", "ence", "dom", "er",
    "or", "ess", "est", "hood", "ian", "ate", "en", "ful", "ive", "ward", "wise",
    "y", "ed", "ing", "es", "est", "ese", "ette", "let", "ling", "logy", "ology",
    "most", "less", "ous", "eous", "ious"
]])

#vocab.add_tokens(VOCAB_ASCII)
#vocab.add_tokens(VOCAB_INTEGERS)
PAD_ID = vocab.PAD_ID
EOS_ID = vocab.EOS_ID
SEP_ID = vocab.token_to_id("<|SEP|>")
VOCAB_SIZE = vocab.vocab_size

device = get_device()
vocab.token_map

In [ ]:
specs = [
    DatasetSpec(source="seastar105/emo_speech_caption_test", split="train", field_map={"text": "transcript", "audio": "audio"}),
    DatasetSpec(source="ylacombe/podcast_fillers_processed", split="train", field_map={"text": "text", "audio": "audio"}),
    DatasetSpec(source="ahazeemi/librispeech10h", split="train.10", field_map={"text": "text", "audio": "audio"}),
]

ds = MultiModalAudioText(specs)

print("_HAS_TA:", _HAS_TA)
print("torchaudio:", torchaudio.__version__ if _HAS_TA else None)

di, (ds0, fm0) = 0, ds.datasets[0]
row0 = ds0[0]
raw = get(row0, fm0.get("audio"))

import io
audio = to_audio_wave(raw)
print("decoded?", audio is not None, "sr:", None if audio is None else audio[1],
      "shape:", None if audio is None else tuple(audio[0].shape))


print("DEVICE:", device, "| len(ds):", len(ds))

In [ ]:
codec = AudioCodec(D=256, hop=128, K=16, n_codes=128, beta=0.25, drop=0.1)
#codec = AudioCodec(D=64, hop=256, K=16, n_codes=128, beta=0.25, drop=0.1)
#codec = AudioCodec(D=128, hop=256, K=4, n_codes=512, beta=0.25, drop=0.1).to(DEVICE)
#codec.load_state_dict(torch.load("audio.codec.pt", map_location="cpu"))# save 64, 128, 16, 512
load_partial_state_dict(codec,f"../artifact/audio.codec.lg.pt", map_location=torch.device('cpu'))

codec = codec.to(device)

print_model_params_count(codec)

In [ ]:
#torch.save(codec.state_dict(), "../artifact/audio.codec.pt")  # save

In [ ]:
import math
from dataclasses import dataclass
from typing import Optional, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F


def valid_to_kpm(valid_mask: torch.Tensor) -> torch.Tensor:
    """
    Convert "valid mask" to PyTorch key_padding_mask.
      valid_mask: (B,L) bool, True means VALID token
      returns:    (B,L) bool, True means PAD (masked out in attention)
    """
    assert valid_mask.dtype == torch.bool
    return ~valid_mask


def make_causal_mask(L: int, device) -> torch.Tensor:
    """(L,L) bool, True means block attention (upper triangular)."""
    return torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)


# -----------------------------
# modules
# -----------------------------
class PosEnc(nn.Module):
    def __init__(self, d: int, max_len: int = 4096):
        super().__init__()
        self.d = d
        self.register_buffer("pe", self._build(max_len, d).unsqueeze(0), persistent=False)

    @staticmethod
    def _build(max_len: int, d: int, device=None, dtype=None):
        pe = torch.zeros(max_len, d, device=device, dtype=dtype)
        pos = torch.arange(0, max_len, device=device, dtype=dtype).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2, device=device, dtype=dtype) * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,L,d)
        L = x.size(1)
        if L > self.pe.size(1):
            # grow on the fly
            self.pe = self._build(L, self.d, device=x.device, dtype=x.dtype).unsqueeze(0)
        return x + self.pe[:, :L]


class TextEncoder(nn.Module):
    def __init__(self, vocab_size: int, d: int, pad_id: int, drop: float):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d, padding_idx=pad_id)
        self.ln = nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        return self.drop(self.ln(self.emb(ids)))


class CodecTokenEmbedConcat(nn.Module):
    """
    codes: (B,K,T) -> (B,T,d) by CONCAT (preserves codebook identity) then project.
    Masking is in embedding-space via learned per-codebook mask vectors.
    """
    def __init__(self, K: int, n_codes: int, d: int, drop: float):
        super().__init__()
        self.K = K
        self.n_codes = n_codes
        self.d = d

        # per-codebook embedding size
        dk = d  # keep same per-book dim for simplicity; projection will reduce
        self.dk = dk

        self.embs = nn.ModuleList([nn.Embedding(n_codes, dk) for _ in range(K)])
        self.mask_emb = nn.Parameter(torch.empty(K, dk))
        nn.init.normal_(self.mask_emb, mean=0.0, std=0.02)

        self.proj = nn.Linear(K * dk, d, bias=False)
        self.ln = nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)

    def forward(self, codes: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        codes: (B,K,T) long
        mask:  (B,T) bool, True means MASK this position (replace with mask vector) for ALL codebooks
        """
        B, K, T = codes.shape
        assert K == self.K

        chunks = []
        for k in range(K):
            ek = self.embs[k](codes[:, k, :])  # (B,T,dk)
            if mask is not None:
                mk = self.mask_emb[k].view(1, 1, -1).expand(B, T, -1)
                ek = torch.where(mask.unsqueeze(-1), mk, ek)
            chunks.append(ek)

        x = torch.cat(chunks, dim=-1)  # (B,T,K*dk)
        x = self.proj(x)               # (B,T,d)
        return self.drop(self.ln(x))


class LengthCond(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2, d),
            nn.GELU(),
            nn.Linear(d, d),
        )

    def forward(self, tok_len: torch.Tensor, sec: torch.Tensor) -> torch.Tensor:
        # tok_len, sec: (B,)
        z = torch.stack(
            [
                torch.log(tok_len.float().clamp_min(1.0)),
                torch.log(sec.float().clamp_min(1e-3)),
            ],
            dim=-1,
        )  # (B,2)
        return self.mlp(z).unsqueeze(1)  # (B,1,d)


@dataclass
class MaskGITSchedule:
    steps: int = 12
    # confidence thresholding:
    # we commit a fraction each step; remaining stay masked
    min_commit: float = 0.10
    max_commit: float = 0.90
    # temperature on logits during sampling (optional)
    temp: float = 1.0
    # use sampling vs argmax
    sample: bool = False
    # if sampling, use top-k (0 = off)
    topk: int = 0


# -----------------------------
# the rewritten denoiser model
# -----------------------------
class TextCondCodecTokenDenoiser(nn.Module):
    """
    Fixes included:
      - Proper MaskGIT-style inference for t2a (iterative unmasking).
      - Training loss should be computed ONLY on masked positions.
      - Include full-mask cases (mask_ratio can include 1.0).
      - Codec embedding changed from sum -> concat+proj (preserves codebook identity).
      - Optional: prevent "copy-unmasked" shortcut by allowing all-mask cases + masked-only loss.
        (Bidirectional self-attn is fine for MaskGIT; don’t use causal in t2a.)
      - a2t generation: uses BOS, not PAD.
    """

    def __init__(
        self,
        vocab_size: int,
        pad_id: int,
        bos_id: int,
        eos_id: int,
        K: int,
        n_codes: int,
        d: int = 256,
        n_heads: int = 8,
        n_text_layers: int = 4,
        n_den_layers: int = 8,
        n_asr_layers: int = 4,
        drop: float = 0.1,
        max_pos: int = 4096,
        # voice conditioning (optional)
        n_voice_classes: int = 16,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.K = K
        self.n_codes = n_codes
        self.d = d

        self.pos = PosEnc(d, max_len=max_pos)

        # ---- text side ----
        self.tenc = TextEncoder(vocab_size, d, pad_id, drop)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=n_heads,
            dim_feedforward=4 * d,
            dropout=drop,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.text_enc = nn.TransformerEncoder(enc_layer, num_layers=n_text_layers)

        # voice + length conditioning appended into memory
        self.voice_emb = nn.ModuleList([nn.Embedding(n_voice_classes, d), nn.Embedding(n_voice_classes, d)])
        self.len_cond = LengthCond(d)

        # ---- NEW: predict log(tok_len) and log(sec) from (text + voice) without alignment ----
        self.len_pred = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, 2),  # [log_tok_len, log_sec]
        )

        # ---- token embedding (concat+proj) ----
        self.tok_emb = CodecTokenEmbedConcat(K, n_codes, d, drop)

        # ---- text -> audio denoiser ----
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d,
            nhead=n_heads,
            dim_feedforward=4 * d,
            dropout=drop,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.den = nn.TransformerDecoder(dec_layer, num_layers=n_den_layers)
        self.tok_out = nn.ModuleList([nn.Linear(d, n_codes) for _ in range(K)])

        # ---- audio(tokens) -> text head (optional ASR) ----
        self.audio_enc = nn.TransformerEncoder(enc_layer, num_layers=n_asr_layers)
        self.text_dec = nn.TransformerDecoder(dec_layer, num_layers=n_asr_layers)
        self.text_out = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, vocab_size, bias=False))

        self.drop = nn.Dropout(drop)
        self.apply(init)

    def _voice_vec(self, voice: Optional[torch.Tensor], B: int, device) -> Optional[torch.Tensor]:
        if voice is None:
            return None
        v = voice.to(device=device, dtype=torch.long)
        v0 = self.voice_emb[0](v[:, 0])
        v1 = self.voice_emb[1](v[:, 1])
        return (v0 + v1).unsqueeze(1)  # (B,1,d)

    # -------- text encoding --------
    def encode_text(
        self,
        input_ids: torch.Tensor,
        attn_mask_valid: torch.Tensor,
        tok_len: torch.Tensor,
        sec: torch.Tensor,
        voice: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        input_ids: (B,L)
        attn_mask_valid: (B,L) bool True=valid token, False=pad
        """
        x = self.pos(self.tenc(input_ids))  # (B,L,d)
        vv = self._voice_vec(voice, input_ids.size(0), x.device)
        if vv is not None:
            x = x + vv

        # append length token to memory
        lc = self.len_cond(tok_len=tok_len, sec=sec)  # (B,1,d)
        x = torch.cat([x, lc], dim=1)  # (B,L+1,d)

        # extend valid mask for length token (always valid)
        attn_mask_valid = torch.cat(
            [attn_mask_valid, torch.ones(attn_mask_valid.size(0), 1, device=attn_mask_valid.device, dtype=torch.bool)],
            dim=1,
        )  # (B,L+1)

        mem_kpm = valid_to_kpm(attn_mask_valid)  # True=pad
        mem = self.text_enc(x, src_key_padding_mask=mem_kpm)
        return mem, mem_kpm

    def _pool_text(self, x_pre: torch.Tensor, attn_mask_valid: torch.Tensor) -> torch.Tensor:
        """
        x_pre: (B,L,d) embeddings BEFORE appending length token, already pos+voice applied
        attn_mask_valid: (B,L) bool True=valid
        returns: (B,d) masked mean pool
        """
        m = attn_mask_valid.unsqueeze(-1).to(dtype=x_pre.dtype)          # (B,L,1)
        denom = m.sum(dim=1).clamp_min(1.0)                              # (B,1)
        return (x_pre * m).sum(dim=1) / denom                            # (B,d)


    def predict_len_sec(
        self,
        input_ids: torch.Tensor,
        attn_mask_valid: torch.Tensor,
        voice: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Predicts (log_tok_len, log_sec) from text+voice.
        Returns:
          log_tok_len: (B,)
          log_sec:     (B,)
          pooled:      (B,d) pooled representation used (for debugging)
        """
        x_pre = self.pos(self.tenc(input_ids))  # (B,L,d)

        vv = self._voice_vec(voice, input_ids.size(0), x_pre.device)
        if vv is not None:
            x_pre = x_pre + vv

        pooled = self._pool_text(x_pre, attn_mask_valid)                 # (B,d)
        pred = self.len_pred(pooled)                                     # (B,2)
        log_tok_len = pred[:, 0]
        log_sec = pred[:, 1]
        return log_tok_len, log_sec, pooled


    def loss_len_sec(
        self,
        log_tok_len_pred: torch.Tensor,
        log_sec_pred: torch.Tensor,
        tok_len_true: torch.Tensor,
        sec_true: torch.Tensor,
    ) -> torch.Tensor:
        """
        Supervise predicted logs with MSE (stable scale).
        """
        # tok_len_true: (B,) long, sec_true: (B,) float
        log_tok_len_true = torch.log(tok_len_true.float().clamp_min(1.0))
        log_sec_true = torch.log(sec_true.float().clamp_min(1e-3))
        return F.mse_loss(log_tok_len_pred, log_tok_len_true) + F.mse_loss(log_sec_pred, log_sec_true)


    @torch.no_grad()
    def generate_audio_tokens_autolen(
        self,
        input_ids: torch.Tensor,
        attn_mask_valid: torch.Tensor,
        codec_hop: int,
        audio_sr: int,
        voice: Optional[torch.Tensor] = None,
        max_tok_len: int = 4096,
        schedule: MaskGITSchedule = MaskGITSchedule(),
    ) -> torch.Tensor:
        """
        NEW: text->audio token generation WITHOUT passing target_sec/tok_len.
        Predict tok_len & sec from text+voice, derive tok_mask, then run MaskGIT.
        """
        self.eval()

        log_tok_len, log_sec, _ = self.predict_len_sec(input_ids, attn_mask_valid, voice=voice)

        # tok_len prediction -> integer frames
        tok_len = torch.exp(log_tok_len).clamp(1.0, float(max_tok_len)).round().long()  # (B,)

        # derive sec; prefer predicted sec but also keep consistent with tok_len/hop/sr
        # (you can swap which you trust more later; this keeps both learned and consistent)
        sec_from_tok = tok_len.float() * float(codec_hop) / float(audio_sr)
        sec_pred = torch.exp(log_sec).clamp_min(1e-3)
        sec = 0.5 * sec_from_tok + 0.5 * sec_pred  # simple blend

        # build tok_mask_valid from tok_len
        B = input_ids.size(0)
        T = int(tok_len.max().item())
        tok_mask_valid = (torch.arange(T, device=input_ids.device).unsqueeze(0) < tok_len.unsqueeze(1))  # (B,T)

        return self.generate_audio_tokens(
            input_ids=input_ids,
            attn_mask_valid=attn_mask_valid,
            tok_mask_valid=tok_mask_valid,
            tok_len=tok_len,
            sec=sec,
            voice=voice,
            schedule=schedule,
        )

    # -------- audio(tokens) encoding (for a2t) --------
    def encode_audio_tokens(self, codes: torch.Tensor, tok_mask_valid: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        codes: (B,K,T)
        tok_mask_valid: (B,T) bool True=valid audio token, False=pad
        """
        x = self.pos(self.tok_emb(codes, mask=None))  # (B,T,d)
        kpm = valid_to_kpm(tok_mask_valid)
        mem = self.audio_enc(x, src_key_padding_mask=kpm)  # (B,T,d)
        return mem, kpm

    # -------- core denoiser forward (t2a) --------
    def forward_t2a(
        self,
        input_ids: torch.Tensor,
        attn_mask_valid: torch.Tensor,
        codes: torch.Tensor,
        tok_mask_valid: torch.Tensor,
        tok_len: torch.Tensor,
        sec: torch.Tensor,
        voice: Optional[torch.Tensor] = None,
        mask_ratio: Tuple[float, float] = (0.10, 1.00),  # IMPORTANT: include 1.0 cases
        force_full_mask_prob: float = 0.10,
    ) -> Dict[str, torch.Tensor]:
        """
        Returns:
          logits_tok: (B,K,T,n_codes)
          mask_m:     (B,T) bool (True=masked positions)
        """
        B, K, T = codes.shape
        assert K == self.K

        mem, mem_kpm = self.encode_text(input_ids, attn_mask_valid, tok_len=tok_len, sec=sec, voice=voice)

        # sample a per-example mask ratio
        lo, hi = float(mask_ratio[0]), float(mask_ratio[1])
        r = torch.empty(B, device=codes.device).uniform_(lo, hi)

        # sometimes force full-mask to prevent copy shortcut
        if force_full_mask_prob > 0:
            full = torch.rand(B, device=codes.device) < force_full_mask_prob
            r = torch.where(full, torch.ones_like(r), r)

        u = torch.rand(B, T, device=codes.device)
        m = (u < r.unsqueeze(1)) & tok_mask_valid  # (B,T) True=mask

        x = self.pos(self.tok_emb(codes, mask=m))  # (B,T,d)

        tgt_kpm = valid_to_kpm(tok_mask_valid)
        h = self.den(
            x, mem,
            tgt_key_padding_mask=tgt_kpm,
            memory_key_padding_mask=mem_kpm,
        )  # (B,T,d)

        logits = []
        hh = self.drop(h)
        for k in range(K):
            logits.append(self.tok_out[k](hh).unsqueeze(1))  # (B,1,T,n_codes)
        logits = torch.cat(logits, dim=1)  # (B,K,T,n_codes)

        return {"logits_tok": logits, "mask_m": m}

    # -------- masked-only CE loss (t2a) --------
    def loss_t2a_masked_only(
        self,
        logits_tok: torch.Tensor,  # (B,K,T,N)
        codes: torch.Tensor,       # (B,K,T)
        tok_mask_valid: torch.Tensor,  # (B,T) valid audio positions
        mask_m: torch.Tensor,      # (B,T) masked positions
    ) -> torch.Tensor:
        """
        Compute CE only over masked positions (and valid positions).
        """
        B, K, T, N = logits_tok.shape
        assert K == self.K and N == self.n_codes

        # positions to score
        score = (mask_m & tok_mask_valid)  # (B,T)
        denom = score.sum().clamp_min(1)

        loss = 0.0
        for k in range(K):
            # (B,T,N) vs (B,T)
            l = F.cross_entropy(
                logits_tok[:, k].reshape(B * T, N),
                codes[:, k].reshape(B * T),
                reduction="none",
            ).reshape(B, T)
            loss = loss + (l * score.float()).sum()

        return loss / denom / K

    # -------- MaskGIT-style generation for t2a --------
    @torch.no_grad()
    def generate_audio_tokens(
        self,
        input_ids: torch.Tensor,
        attn_mask_valid: torch.Tensor,
        tok_mask_valid: torch.Tensor,
        tok_len: torch.Tensor,
        sec: torch.Tensor,
        voice: Optional[torch.Tensor] = None,
        schedule: MaskGITSchedule = MaskGITSchedule(),
    ) -> torch.Tensor:
        """
        Returns:
          codes: (B,K,T) generated codec tokens
        """
        self.eval()
        device = input_ids.device
        B, T = tok_mask_valid.shape
        K = self.K

        mem, mem_kpm = self.encode_text(input_ids, attn_mask_valid, tok_len=tok_len, sec=sec, voice=voice)

        # initialize: all masked positions set to 0 (any id is fine; embeddings get replaced by mask vec)
        codes = torch.zeros(B, K, T, device=device, dtype=torch.long)
        masked = tok_mask_valid.clone()  # start with all valid positions masked

        # helper: compute per-position confidence
        def conf_from_logits(logits_k: torch.Tensor) -> torch.Tensor:
            # logits_k: (B,T,N)
            probs = F.softmax(logits_k / max(schedule.temp, 1e-6), dim=-1)
            conf, _ = probs.max(dim=-1)  # (B,T)
            return conf

        # commit fraction ramp
        steps = max(int(schedule.steps), 1)
        for s in range(steps):
            # predict all positions given current partially-filled codes; masked positions use mask embeddings
            x = self.pos(self.tok_emb(codes, mask=masked))  # (B,T,d)
            tgt_kpm = valid_to_kpm(tok_mask_valid)

            h = self.den(
                x, mem,
                tgt_key_padding_mask=tgt_kpm,
                memory_key_padding_mask=mem_kpm,
            )  # (B,T,d)

            # logits per codebook
            logits_per_k = [self.tok_out[k](h) for k in range(K)]  # each (B,T,N)

            # choose candidate tokens
            new_codes = []
            new_confs = []
            for k in range(K):
                logits_k = logits_per_k[k]  # (B,T,N)
                if schedule.sample:
                    probs = F.softmax(logits_k / max(schedule.temp, 1e-6), dim=-1)
                    if schedule.topk and schedule.topk > 0:
                        topk = min(schedule.topk, probs.size(-1))
                        vals, idx = torch.topk(probs, topk, dim=-1)
                        vals = vals / vals.sum(dim=-1, keepdim=True)
                        # sample among topk
                        pick = torch.multinomial(vals.view(B * T, topk), 1).view(B, T, 1)
                        tok = torch.gather(idx, -1, pick).squeeze(-1)  # (B,T)
                    else:
                        tok = torch.multinomial(probs.view(B * T, -1), 1).view(B, T)
                else:
                    tok = torch.argmax(logits_k, dim=-1)  # (B,T)

                new_codes.append(tok)
                new_confs.append(conf_from_logits(logits_k))

            # aggregate confidence across codebooks (min is conservative)
            conf = torch.stack(new_confs, dim=0).min(dim=0).values  # (B,T)

            # decide how many masked positions to commit this step
            # fraction ramps from min_commit -> max_commit over steps
            frac = schedule.min_commit + (schedule.max_commit - schedule.min_commit) * (s / max(steps - 1, 1))
            frac = float(min(max(frac, 0.0), 1.0))

            # commit highest-confidence masked positions per sample
            for b in range(B):
                mb = masked[b] & tok_mask_valid[b]  # positions still masked
                n_left = int(mb.sum().item())
                if n_left == 0:
                    continue
                n_commit = max(1, int(math.ceil(frac * n_left))) if s < steps - 1 else n_left

                # pick indices among masked positions with highest confidence
                conf_b = conf[b].clone()
                conf_b[~mb] = -1.0
                top_idx = torch.topk(conf_b, k=n_commit, dim=0).indices  # (n_commit,)

                # write tokens for all K at those positions
                for k in range(K):
                    codes[b, k, top_idx] = new_codes[k][b, top_idx]

                # unmask committed positions
                masked[b, top_idx] = False

        # make sure invalid pads are zeroed (optional hygiene)
        codes = codes * tok_mask_valid.unsqueeze(1).long()
        return codes

    # -------- a2t (audio->text) forward --------
    def forward_a2t(self, input_ids: torch.Tensor, tok_mask_valid: torch.Tensor, codes: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Teacher-forced: predict next token from shifted-right decoder input.
        input_ids contains BOS...EOS...PAD
        """
        mem_a, mem_kpm = self.encode_audio_tokens(codes, tok_mask_valid)

        # shift-right with BOS
        dec_in = torch.full_like(input_ids, self.pad_id)
        dec_in[:, 0] = self.bos_id
        dec_in[:, 1:] = input_ids[:, :-1]

        tgt = self.pos(self.tenc(dec_in))
        L = tgt.size(1)
        causal = make_causal_mask(L, tgt.device)

        tgt_kpm = (dec_in == self.pad_id)
        tgt_kpm[:, 0] = False  # keep BOS visible

        out = self.text_dec(
            tgt, mem_a,
            tgt_mask=causal,
            tgt_key_padding_mask=tgt_kpm,
            memory_key_padding_mask=mem_kpm,
        )
        logits_text = self.text_out(self.drop(out))
        return {"logits_text": logits_text}

    # -------- a2t generation --------
    @torch.no_grad()
    def generate_text(self, codes: torch.Tensor, tok_mask_valid: torch.Tensor, max_len: int = 128) -> torch.Tensor:
        self.eval()
        mem_a, mem_kpm = self.encode_audio_tokens(codes, tok_mask_valid)
        B = codes.size(0)
        device = codes.device

        ids = torch.full((B, 1), self.bos_id, device=device, dtype=torch.long)

        for _ in range(max_len):
            tgt = self.pos(self.tenc(ids))
            L = tgt.size(1)
            causal = make_causal_mask(L, device)

            tgt_kpm = (ids == self.pad_id)
            tgt_kpm[:, 0] = False  # BOS visible

            out = self.text_dec(
                tgt, mem_a,
                tgt_mask=causal,
                tgt_key_padding_mask=tgt_kpm,
                memory_key_padding_mask=mem_kpm,
            )
            nxt = torch.argmax(self.text_out(out[:, -1]), dim=-1, keepdim=True)
            ids = torch.cat([ids, nxt], dim=1)
            if (nxt.squeeze(1) == self.eos_id).all():
                break

        return ids[:, 1:]  # drop BOS

In [ ]:
# =========================
# Cell 1 — Instantiate model, optimizer, loaders
# =========================
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

# ---- assume you already have:
#   - TextVocabulary as `vocab` (with PAD_ID, BOS_ID, EOS_ID, UNK_ID)
#   - dataset = MultiModalAudioText(specs)
#   - collate = make_collate_audio_text(vocab, ...)
#   - codec = AudioCodec(...).to(device)
#   - denoiser class = TextCondCodecTokenDenoiser (from previous message)


# DataLoader
collate_fn = make_collate_audio_text(vocab, max_text_len=DEFAULT_TEXT_LENGTH_TOKENS, audio_sr=SAMPLE_RATE, audio_len=DEFAULT_AUDIO_LENGTH_SECONDS)
train_dl = DataLoader(
    ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=(device == "cuda"),
    collate_fn=collate_fn,
    drop_last=True,
)

# Instantiate denoiser
K = codec.K
n_codes = codec.n_codes

denoiser = TextCondCodecTokenDenoiser(
    vocab_size=vocab.vocab_size,
    pad_id=vocab.PAD_ID,
    bos_id=vocab.BOS_ID,
    eos_id=vocab.EOS_ID,
    K=K,
    n_codes=n_codes,
    d=256,
    n_heads=8,
    n_text_layers=4,
    n_den_layers=8,
    n_asr_layers=4,
    drop=0.1,
    max_pos=4096,
)
#load_partial_state_dict(codec,f"../artifact/audio.denoiser.pt", map_location=torch.device('cpu'))
denoiser = denoiser.to(device)


print("device:", device)
print("denoiser params:", sum(p.numel() for p in denoiser.parameters())/1e6, "M")

In [ ]:
# =========================
# Cell 2 — Training loop (t2a MaskGIT training) with tqdm
# =========================
from dataclasses import dataclass


# Optimizer + AMP
lr = 3e-5
decay, no_decay = [], []
for name, p in denoiser.named_parameters():
    if not p.requires_grad:
        continue

    # no weight decay for biases, LayerNorm, Embeddings (incl. positional / token emb)
    if (
        name.endswith(".bias")
        or ".ln." in name
        or "layernorm" in name.lower()
        or "norm" in name.lower()
        or "emb" in name.lower()
        or "embedding" in name.lower()
    ):
        no_decay.append(p)
    else:
        decay.append(p)

opt = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.1},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=lr,
    betas=(0.9, 0.98),
)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

# If codec is already trained/frozen, do this:
codec.eval()
for p in codec.parameters():
    p.requires_grad = False


# =========================
# Cell 2 — Training loop (t2a + a2t) with instant tqdm losses
# =========================
from dataclasses import dataclass
import torch.nn.functional as F
from tqdm import tqdm

@dataclass
class TrainCfg:
    epochs: int = 5
    grad_clip: float = 1.0

    # t2a masking
    mask_ratio: tuple = (0.10, 1.00)
    force_full_mask_prob: float = 0.10

    # loss weights
    w_t2a: float = 1.0
    w_a2t: float = 0.5
    w_len: float = 0.1

    log_every: int = 1
    clean_every: int = 100
    save_every: int = 1000

cfg = TrainCfg()

denoiser.train()
global_step = 0

for epoch in range(cfg.epochs):
    pbar = tqdm(train_dl, desc=f"epoch {epoch+1}/{cfg.epochs}", leave=True, position=0)
    for batch in pbar:
        # ---- move batch to device ----
        input_ids = batch["input_ids"].to(device, non_blocking=True)               # (B,L) includes BOS/EOS/PAD
        attn_mask_valid = batch["attn_mask"].to(device, non_blocking=True)         # (B,L) True=valid
        wav = batch["wav"].to(device, non_blocking=True)                           # (B,T)
        wav_len = batch["wav_len"].to(device, non_blocking=True)                   # (B,)
        voice_ids = batch["voice_ids"].to(device, non_blocking=True)               # (B,2)

        has_text = batch["has_text"].to(device, non_blocking=True)
        has_audio = batch["has_audio"].to(device, non_blocking=True)
        keep = has_text & has_audio
        if keep.sum().item() == 0:
            continue

        input_ids = input_ids[keep]
        attn_mask_valid = attn_mask_valid[keep]
        wav = wav[keep]
        wav_len = wav_len[keep]
        voice_ids = voice_ids[keep]

        # ---- compute codec codes ----
        with torch.no_grad():
            codes = codec.encode_codes(wav)  # (B,K,Tc)
            B, K, Tc = codes.shape

        hop = int(codec.hop)
        tc_len = torch.clamp((wav_len + hop - 1) // hop, min=1, max=Tc)  # (B,)
        tok_mask_valid = (torch.arange(Tc, device=device).unsqueeze(0) < tc_len.unsqueeze(1))  # (B,Tc)

        tok_len = tc_len
        sec = wav_len.float() / float(batch["audio_sr"])

        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            # -------- t2a --------
            out_t2a = denoiser.forward_t2a(
                input_ids=input_ids,
                attn_mask_valid=attn_mask_valid,
                codes=codes,
                tok_mask_valid=tok_mask_valid,
                tok_len=tok_len,
                sec=sec,
                voice=voice_ids,
                mask_ratio=cfg.mask_ratio,
                force_full_mask_prob=cfg.force_full_mask_prob,
            )
            loss_t2a = denoiser.loss_t2a_masked_only(
                logits_tok=out_t2a["logits_tok"],
                codes=codes,
                tok_mask_valid=tok_mask_valid,
                mask_m=out_t2a["mask_m"],
            )

            # -------- a2t --------
            # teacher forced next-token prediction, compute CE on valid (non-pad) targets
            out_a2t = denoiser.forward_a2t(
                input_ids=input_ids,
                tok_mask_valid=tok_mask_valid,
                codes=codes,
            )
            logits_text = out_a2t["logits_text"]  # (B,L,V)
            targets = input_ids.clone()           # (B,L)

            # Always ignore PAD
            targets[targets == vocab.PAD_ID] = -100

            # Also ignore BOS targets (recommended)
            targets[targets == vocab.BOS_ID] = -100

            # (Optional) ignore EOS too:
            # targets[targets == vocab.EOS_ID] = -100

            loss_a2t = F.cross_entropy(
                logits_text.reshape(-1, logits_text.size(-1)),
                targets.reshape(-1),
                ignore_index=-100,
            )

            log_tok_len_pred, log_sec_pred, _ = denoiser.predict_len_sec(
                input_ids=input_ids,
                attn_mask_valid=attn_mask_valid,
                voice=voice_ids,
            )
            loss_len = denoiser.loss_len_sec(
                log_tok_len_pred=log_tok_len_pred,
                log_sec_pred=log_sec_pred,
                tok_len_true=tok_len,
                sec_true=sec,
            )

            loss = cfg.w_t2a * loss_t2a + cfg.w_a2t * loss_a2t + getattr(cfg, "w_len", 0.1) * loss_len

        scaler.scale(loss).backward()
        if cfg.grad_clip and cfg.grad_clip > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(denoiser.parameters(), cfg.grad_clip)

        scaler.step(opt)
        scaler.update()

        global_step += 1

        # ---- instantaneous stats for tqdm ----
        m = out_t2a["mask_m"]
        masked_frac = (m & tok_mask_valid).float().sum() / tok_mask_valid.float().sum().clamp_min(1.0)
        Tc_mean = float(tc_len.float().mean().item())

        if global_step % cfg.save_every == 0:
            torch.save(codec.state_dict(), "../artifact/audio.denoiser.pt")

        if global_step % cfg.clean_every == 0:
            cleanup()

        if global_step % cfg.log_every == 0:
            pbar.set_postfix(
                step=global_step,
                tot=f"{loss.item():.4f}",
                t2a=f"{loss_t2a.item():.4f}",
                a2t=f"{loss_a2t.item():.4f}",
                len=f"{loss_len.item():.4f}",
                m_pct=f"{(100.0*masked_frac.item()):.1f}",
                Tc=f"{Tc_mean:.1f}",
            )

In [ ]:
plot_model_matrices(denoiser)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

def infer_t2a_and_render(
    text: str,
    voice_ids_pair=(10, 10),
    schedule=None,
    audio_sr: int = SAMPLE_RATE,
    max_text_len: int = DEFAULT_TEXT_LENGTH_TOKENS,
    max_tok_len: int = 4096,
):
    denoiser.eval()
    codec.eval()

    # ---- build text ids with BOS/EOS ----
    ids = vocab.tokenize(text)
    ids = ids[: max_text_len - 2]                 # reserve BOS+EOS
    ids = [vocab.BOS_ID] + ids
    if ids[-1] != vocab.EOS_ID:
        ids.append(vocab.EOS_ID)

    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)  # (1,L)
    attn_mask_valid = torch.ones_like(input_ids, dtype=torch.bool)               # (1,L)

    voice_ids = torch.tensor([list(voice_ids_pair)], device=device, dtype=torch.long)

    if schedule is None:
        schedule = MaskGITSchedule(
            steps=12,
            min_commit=0.05,
            max_commit=0.85,
            temp=1.3,
            sample=True,
            topk=0,
        )

    hop = int(codec.hop)

    with torch.no_grad():
        # ---- NEW: model predicts tok_len/sec internally ----
        codes_hat = denoiser.generate_audio_tokens_autolen(
            input_ids=input_ids,
            attn_mask_valid=attn_mask_valid,
            codec_hop=hop,
            audio_sr=int(audio_sr),
            voice=voice_ids,
            max_tok_len=max_tok_len,
            schedule=schedule,
        )  # (1,K,Tc_pred)

        wav_hat = codec.decode_codes(codes_hat)    # (1,T')

    wav = wav_hat.squeeze(0).detach().cpu().float().numpy()

    # ---- display audio ----
    display(Audio(wav, rate=int(audio_sr)))

    # ---- plot waveform ----
    t = np.arange(wav.shape[0]) / float(audio_sr)
    plt.figure()
    plt.plot(t, wav)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title("Generated waveform")
    plt.show()

    Tc = codes_hat.size(-1)
    sec_pred = (Tc * hop) / float(audio_sr)

    print(f"text ids length: {input_ids.size(1)}")
    print(f"audio_sr: {audio_sr} | hop: {hop}")
    print(f"Tc_pred (codec frames): {Tc} | sec_pred≈ {sec_pred:.3f}")
    print(f"codes_hat: {tuple(codes_hat.shape)} | wav_hat: {tuple(wav_hat.shape)}")

    return wav_hat, codes_hat

# Example:
wav_hat, codes_hat = infer_t2a_and_render("HELLO WORLD, I LOVE MY LIFE!", voice_ids_pair=(10, 10))

In [ ]:
# =========================
# Cell — a2t inference demo (codec tokens -> text)
# =========================
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

denoiser.eval()
codec.eval()

batch = next(iter(train_dl))

wav = batch["wav"].to(device, non_blocking=True)          # (B,T)
wav_len = batch["wav_len"].to(device, non_blocking=True)
has_audio = batch["has_audio"].to(device, non_blocking=True)

# pick first example with audio
idx = int(torch.nonzero(has_audio, as_tuple=False)[0].item())

wav1 = wav[idx:idx+1]
wav_len1 = wav_len[idx:idx+1]

# -------------------------
# display ground truth audio
# -------------------------
audio_sr = batch["audio_sr"]
gt = wav1[0, : wav_len1.item()].detach().cpu().numpy()

print("---- ground truth audio ----")
display(Audio(gt, rate=audio_sr))

t = np.arange(len(gt)) / audio_sr
plt.figure()
plt.plot(t, gt)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Ground Truth Waveform")
plt.show()

# -------------------------
# Encode codec tokens
# -------------------------
with torch.no_grad():
    codes = codec.encode_codes(wav1)  # (1,K,Tc)

hop = int(codec.hop)
B, K, Tc = codes.shape

tc_len = torch.clamp((wav_len1 + hop - 1) // hop, min=1, max=Tc)
tok_mask_valid = (
    torch.arange(Tc, device=device).unsqueeze(0) < tc_len.unsqueeze(1)
)

# -------------------------
# Run a2t generation
# -------------------------
with torch.no_grad():
    pred_ids = denoiser.generate_text(
        codes=codes,
        tok_mask_valid=tok_mask_valid,
        max_len=192
    )

pred_ids_list = pred_ids[0].detach().cpu().tolist()
pred_text = vocab.detokenize(pred_ids_list, clean=True)

print("\n---- a2t prediction ----")
print("pred token ids:", pred_ids_list[:50], "..." if len(pred_ids_list) > 50 else "")
print("pred text:", pred_text)

# reference text
if "input_ids" in batch:
    ref_ids = batch["input_ids"][idx].detach().cpu().tolist()
    if vocab.EOS_ID in ref_ids:
        ref_ids = ref_ids[: ref_ids.index(vocab.EOS_ID) + 1]

    print("\n---- reference ----")
    print("ref decoded:", vocab.detokenize(ref_ids, clean=True))

In [ ]:
torch.save(denoiser.state_dict(), "../artifact/audio.denoiser.pt")  # save
